# FloodScope 3D: train the 1-7 day discharge forecast (NeuralHydrology on Caravan)

Run this in **Google Colab with a GPU** (Runtime > Change runtime type > T4 or better).

What it does:
1. Downloads Caravan v1.6 (netCDF, **24.8 GB**, CC BY 4.0, Kratzert et al. 2023) and extracts the chosen sub-datasets.
2. Builds model inputs with `floodscope_ml.prepare` (flow as a ratio to each river's mean, weather and soil, 7 days of lead weather).
3. Trains a NeuralHydrology `cudalstm` (365-day history, predicts days 1-7).
4. Evaluates NSE/KGE on **held-out basins** against persistence and GloFAS, and measures an error band.
5. Exports `forecast_lstm.onnx` + `forecast_lstm.json` for the FastAPI server.

Rough cost on a T4: download + extract 30-60 min, preparation 20-40 min, training 2-4 h for ~2,000 basins.
Keep everything on Google Drive so a disconnect does not lose work.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/floodscope"
import os

os.makedirs(WORK, exist_ok=True)

## Get the FloodScope code
Either clone your repository (set `REPO_URL`), or upload the repo's `ml/` folder to `WORK/ml` with the Files pane.

In [ ]:
REPO_URL = ""  # e.g. 'https://github.com/<you>/floodscope.git'; leave empty if you uploaded ml/
if REPO_URL:
    !git clone --depth 1 {REPO_URL} /content/floodscope
    ML_DIR = "/content/floodscope/ml"
else:
    ML_DIR = f"{WORK}/ml"
!pip install -q -e "{ML_DIR}[train]"
import sys

sys.path.insert(0, ML_DIR)

In [ ]:
SUBDATASETS = ["camels", "camelsgb", "camelsaus", "lamah"]  # ~2,300 basins; add more if time allows
ARCHIVE = f"{WORK}/Caravan-nc.tar.gz"
URL = "https://zenodo.org/records/15529786/files/Caravan-nc.tar.gz?download=1"  # checked 2026-10-24
import os

if not os.path.exists(ARCHIVE):
    !wget -q --show-progress -O {ARCHIVE} "{URL}"
!mkdir -p /content/caravan_raw
patterns = " ".join(f"'*/timeseries/netcdf/{s}/*' '*/attributes/{s}/*'" for s in SUBDATASETS)
!tar -xzf {ARCHIVE} -C /content/caravan_raw --wildcards {patterns}
!find /content/caravan_raw -maxdepth 2 -type d

In [ ]:
# Check the variable names floodscope_ml expects against a real basin before spending hours.
import glob
from pathlib import Path

CARAVAN = Path(glob.glob("/content/caravan_raw/*/timeseries")[0]).parent
from floodscope_ml.prepare import basins_in, check_variables, load_basin

b = basins_in(CARAVAN, SUBDATASETS[0])[0]
df = load_basin(CARAVAN, b)
check_variables(df, b)  # raises with the available names if anything differs
print(b, df.index.min(), df.index.max(), len(df.columns), "columns OK")

In [ ]:
PREPARED = "/content/prepared"
!python -m floodscope_ml.prepare --caravan-dir {CARAVAN} --out-dir {PREPARED} --subdatasets {' '.join(SUBDATASETS)} --test-fraction 0.2

In [ ]:
from floodscope_ml.train import train, write_config

cfg_path = write_config(Path(PREPARED), Path(f"{WORK}/runs"), Path(f"{WORK}/config.yml"))
run_dir = train(cfg_path)
print("run dir:", run_dir)

In [ ]:
# Held-out basins: these never appeared in training.
from neuralhydrology.nh_run import eval_run

eval_run(run_dir=run_dir, period="test")

from floodscope_ml.evaluate import (
    basin_series,
    error_band,
    load_test_results,
    score_basin,
    summarise,
    today_flow,
)

results = load_test_results(run_dir)
test_basins = sorted(results)
scores = []
for b in test_basins:
    scores += score_basin(basin_series(results, b), today_q=today_flow(Path(PREPARED), b))
table = summarise(scores)
band = error_band(results, test_basins)
table

In [ ]:
# Compare with GloFAS on a sample of held-out gauges (Open-Meteo is rate limited, so keep it small).
# This is the GloFAS *reanalysis* (driven by observed weather); archived GloFAS forecasts need a CDS account.
import random
import time

import pandas as pd
import requests

attrs = pd.concat(
    [
        pd.read_csv(f, index_col="gauge_id")
        for f in glob.glob(f"{CARAVAN}/attributes/*/attributes_other_*.csv")
    ]
)
sample = random.Random(0).sample(test_basins, min(50, len(test_basins)))
g_scores = []
for b in sample:
    lat, lon = attrs.loc[b, "gauge_lat"], attrs.loc[b, "gauge_lon"]
    r = requests.get(
        "https://flood-api.open-meteo.com/v1/flood",
        params=dict(
            latitude=lat,
            longitude=lon,
            daily="river_discharge",
            start_date="1997-01-01",
            end_date="2019-12-31",
        ),
        timeout=120,
    ).json()
    q = pd.Series(
        r["daily"]["river_discharge"], index=pd.to_datetime(r["daily"]["time"]), dtype=float
    )
    q_ratio = q / q.loc["1997-01-01":"2010-09-30"].mean()
    g_scores += score_basin(basin_series(results, b), glofas_q=q_ratio)
    time.sleep(2)
glofas_table = summarise(g_scores)
glofas_table

In [ ]:
from floodscope_ml.export import export_run

train_basins = Path(PREPARED, "train_basins.txt").read_text().split()
evaluation = {
    "held_out_basins": len(test_basins),
    "median_by_lead": table.reset_index().to_dict(orient="records"),
    "glofas_reanalysis_sample": glofas_table.reset_index().to_dict(orient="records"),
    "log_ratio_quantiles": band,
    "note": "NSE/KGE on q_ratio for held-out basins. GloFAS comparison uses its reanalysis, not archived forecasts.",
}
training_info = {
    "dataset": "Caravan v1.6 (CC BY 4.0)",
    "subdatasets": SUBDATASETS,
    "train_basins": len(train_basins),
}
onnx_path, meta_path = export_run(run_dir, Path(f"{WORK}/artifacts"), evaluation, training_info)
print(onnx_path, meta_path)
from google.colab import files

files.download(str(onnx_path))
files.download(str(meta_path))

## Use the model in FloodScope
Copy `forecast_lstm.onnx` and `forecast_lstm.json` into the repo at `ml/artifacts/` (or point `FLOODSCOPE_MODEL_DIR` at their folder) and restart the API. `/api/predict` then returns the AI forecast, and the 3D view gets an "AI forecast" tab.